# 🧠 Model Experiments — Comparison & Analysis

Comprehensive model comparison with inline visualizations:
- LightGBM vs XGBoost vs LSTM-AE + Isolation Forest
- Cross-validation results
- Ablation study visualization
- SHAP feature importance
- ROC/PR curves

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import sys, os

ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(ROOT))
os.chdir(str(ROOT))

sns.set_theme(style='darkgrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.family': 'sans-serif'})

from argus.config import Config
Config.setup()

## 1. Load Data & Models

In [ ]:
data_dir = Config.paths.DATA / 'processed'
X = np.load(data_dir / 'X_enhanced.npy')
y = np.load(data_dir / 'y_enhanced.npy')
feature_names = json.loads((data_dir / 'enhanced_feature_cols.json').read_text())

if X.ndim == 3:
    X = X[:, -1, :]
X = np.nan_to_num(X, 0.0)

print(f'Data: X={X.shape}, y={y.shape}')
print(f'Features: {len(feature_names)}')
print(f'Positive rate: {y.mean():.3f} ({int(y.sum())}/{len(y)})')

## 2. Model Training & ROC Curves

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, precision_recall_curve, f1_score, classification_report
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Train models
models = {
    'LightGBM': LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, scale_pos_weight=10,
                                random_state=42, verbose=-1),
    'XGBoost': XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, scale_pos_weight=10,
                              random_state=42, verbosity=0, eval_metric='logloss'),
}

results = {}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = {'LightGBM': '#06b6d4', 'XGBoost': '#8b5cf6'}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=colors[name], lw=2, label=f'{name} (AUC={roc_auc:.4f})')
    
    # PR
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(rec, prec)
    ax2.plot(rec, prec, color=colors[name], lw=2, label=f'{name} (PR-AUC={pr_auc:.4f})')
    
    f1 = f1_score(y_test, y_pred)
    results[name] = {'F1': f1, 'AUC-ROC': roc_auc, 'PR-AUC': pr_auc}
    print(f'{name}: F1={f1:.4f}, AUC={roc_auc:.4f}')

ax1.plot([0,1], [0,1], '--', color='gray', alpha=0.5)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves', fontweight='bold')
ax1.legend()

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Cross-Validation Results

In [ ]:
# Load CV results if available
cv_path = ROOT / 'research' / '11_cross_validation.json'
if cv_path.exists():
    cv_results = json.loads(cv_path.read_text())
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    metrics = ['f1', 'auc', 'precision', 'recall']
    titles = ['F1 Score', 'AUC-ROC', 'Precision', 'Recall']
    
    for i, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[i]
        for j, (model_name, data) in enumerate(cv_results.items()):
            vals = data[metric]['values']
            mean = data[metric]['mean']
            x = np.arange(1, 6)
            ax.bar(x + j*0.35 - 0.175, vals, 0.3, label=model_name,
                   color=['#06b6d4', '#8b5cf6'][j], alpha=0.8)
            ax.axhline(y=mean, color=['#06b6d4', '#8b5cf6'][j], linestyle='--', alpha=0.5)
        ax.set_xlabel('Fold')
        ax.set_title(title, fontweight='bold')
        if i == 0:
            ax.legend()
    
    plt.suptitle('5-Fold Stratified Cross-Validation Results', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Summary table
    summary = []
    for model, data in cv_results.items():
        summary.append({
            'Model': model,
            'F1': f"{data['f1']['mean']:.4f} +/- {data['f1']['std']:.4f}",
            'AUC': f"{data['auc']['mean']:.4f} +/- {data['auc']['std']:.4f}",
            'Precision': f"{data['precision']['mean']:.4f} +/- {data['precision']['std']:.4f}",
            'Recall': f"{data['recall']['mean']:.4f} +/- {data['recall']['std']:.4f}",
            'FPR': f"{data['fpr']['mean']*100:.3f}%",
        })
    display(pd.DataFrame(summary))
else:
    print('Run: python -m argus.experiments.cross_validation first')

## 4. Ablation Study Visualization

In [ ]:
ablation_path = ROOT / 'research' / '12_ablation_study.json'
if ablation_path.exists():
    ablation = json.loads(ablation_path.read_text())
    baseline_f1 = ablation['baseline']['f1']
    
    # Feature ablation chart
    feat_ablation = ablation['feature_ablation']
    cats = sorted(feat_ablation.keys(), key=lambda k: feat_ablation[k]['f1_drop'], reverse=True)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    drops = [feat_ablation[c]['f1_drop'] for c in cats]
    colors_bar = ['#ef4444' if d > 0 else '#22c55e' for d in drops]
    
    bars = ax.barh(range(len(cats)), drops, color=colors_bar, alpha=0.8)
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels([f"{c} ({feat_ablation[c]['num_features_removed']} feats)" for c in cats])
    ax.axvline(x=0, color='white', linewidth=1)
    ax.set_xlabel('F1 Drop (positive = performance decreases when removed)')
    ax.set_title(f'Feature Category Ablation (Baseline F1={baseline_f1:.4f})', fontweight='bold')
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
else:
    print('Run: python -m argus.experiments.ablation_study first')

## 5. SHAP Feature Importance

In [ ]:
try:
    import shap
    
    # Use the LightGBM model trained above
    lgb_model = models['LightGBM']
    
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X_test[:500])  # Sample for speed
    
    # If binary output, take the positive class
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values
    
    # Summary plot
    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_vals, X_test[:500], feature_names=feature_names, 
                      max_display=20, show=False)
    plt.title('SHAP Feature Importance (Top 20)', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Bar plot
    mean_abs_shap = np.abs(shap_vals).mean(axis=0)
    top_idx = np.argsort(mean_abs_shap)[-15:][::-1]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(range(len(top_idx)), mean_abs_shap[top_idx], color='#8b5cf6', alpha=0.8)
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels([feature_names[i] for i in top_idx])
    ax.set_xlabel('Mean |SHAP Value|')
    ax.set_title('Top 15 Features by SHAP Importance', fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('SHAP not installed. Run: pip install shap')
except Exception as e:
    print(f'SHAP error: {e}')

## 6. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Insider'])
    disp.plot(ax=axes[i], cmap='Blues', values_format='d')
    axes[i].set_title(f'{name} Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

# Detailed classification report
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Insider']))

## 7. Summary

| Model | F1 (CV) | AUC (CV) | FPR | Notes |
|-------|---------|----------|-----|-------|
| LightGBM | 0.935 +/- 0.022 | 0.991 +/- 0.002 | 0.03% | Primary model, best individual F1 |
| XGBoost | 0.935 +/- 0.024 | 0.990 +/- 0.006 | 0.04% | Comparable performance, ensemble diversity |
| Meta-Learner | 0.949 | 0.983 | 0.3% | Combines LGB + XGB + LSTM-AE/IF |

### Key Findings
- **Rolling 14-day** features are most important (F1 drops 0.021 when removed)
- **`clearance_normalized`** is the strongest single predictor (SHAP=0.610)
- **FPR < 0.05%** — excellent for banking (< 1 false alarm per 2000 employees per day)
- Both GBDT models are very stable across folds (low variance)